## Handling Long Conversations

We have used seen the concept of short-term memory, which helps your agent keep track of all the messages (Human, AI and Tool) that have been exchanged so far between Human & Agent. We enabled this simply by _attaching_ an instance of `InMemorySaver` class to the `create_agent()` call as shown below.

```python
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="openai:gpt-5-nano",
    ...
    checkpointer=InMemorySaver(),
)
```
With the help of this checkpointer, an Agents is able to keep track of all messages, so it can remember the complete thread of any conversation with the user. This work very well for _short_ conversations. After a while the list becomes longer & longer so that it overflows the size of our Agent's (actually the model of the Agent) context window - recall that each model has a pre-defined size of a context windows (for example GPT models is typically 400K tokens, Haiku is 200K and Sonnet/Open is 1 million and so on). At this point, the model _cannot literally_ parse all this information in an efficient way, thus slowing down our app, impacting performance and increasing its cost.

There are 2 ways we can solve this problem, both using middleware.
1. **Summarizing the conversation** so far, and
2. Trimming or deleting (older) messages

We will cover both these techniques in this workbook.

In [1]:
from dotenv import load_dotenv
from rich.console import Console

load_dotenv(override=True)
console = Console()

## Summarizing the Conversation

Summarizing the conversation can be done using _out-of-the-box_ middleware (viz. `langchain.agents.middleware.SummarizationMiddleware`). The `SummarizationMiddleware` summarizes conversation history when token limits are approached. This middleware monitors message token counts and automatically summarizes older messages when a threshold is reached, preserving recent messages and maintaining context continuity by ensuring AI/Tool message pairs remain together. 

Here is the definition:

```python
SummarizationMiddleware(
    model="openai:gpt-5-nano",
    trigger=...,
    keep=..., 
    token_counter="...", 
    summary_prompt=..., 
    trim_tokens_to_summarize = ...,
)
```

| Parameter | Meaning |
| :-- | :-- |
| `model` | Model used to summarize context - use string similar to the `model` parameter of `create_agent()` call. For example, `openai:gpt-5-nano` (need not be same as the model used by `create_agent()`) | 
| `trigger` | Event (or threshold) that triggers summarization of the context. <br/> Examples: <br/> `("messages", 50)` - fire when 50 messages threshold is reached in agent's checkpointer <br/> `("tokens", 3000)` - fire when 3000 tokens is reached <br/> `[("fraction", 0.8), ("messages", 100)]` - fire when either when 80% of model's max input tokens is reached or when 100 messages is reached (whichever comes first) |
| `keep` | How much of the context to keep. Defaults to keeping the most recent 20 _human_ messages. <br/> Examples: <br/> `("messages", 20)` - keep the most recent 20 _human_ messages (and related AI responses). So I get (<Summary of previous messages> + <20 Human messages + their AI responses>) as my summarized context <br/> `("tokens", 3000)` - keep the most recent 3000 tokens <br/> `("fraction", 0.3)` - Keep the most recent 30% of the model's max input tokens |
| `token_counter` | **(Optional)** - function to count the tokens in message. Defaults to `count_tokens_approximately`|
| `summary_prompt` | **(Optional)** - prompt template for generating summary. Defaults to `DEFAULT_SUMMARY_PROMPT` constant defined by LangChain|
| `trim_tokens_to_summarize` | **(Optional)** - Maximum tokens to keep when preparing messages for the summarization call. Defaults to `_DEFAULT_TRIM_TOKEN_LIMIT` defined internally by LangChain|

In most cases, we'll use the `model`, `triggers` and `keep` parameters leaving others at default values.

An example implementation is shown below. 

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware


agent = create_agent(
    model="openai:gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            # using a different (cheaper) model to summarize
            model="openai:gpt-4o-mini",
            # fire when 100 tokens are reached
            trigger=("tokens", 100),
            # keep the most recent human message + AI response
            keep=("messages", 1),
        )
    ],
)

Let's simulate a large number of messages being passed in the context to our agent. This should (hopefully) trigger our Summarization middleware. We have _deliberately_ kept the thresholds small in the example above. In practice, these would be reasonably larger.

In [ ]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

messages = [
    HumanMessage(content="What is the capital of the moon?"),
    AIMessage(content="The capital of the moon is Lunapolis."),
    HumanMessage(content="What is the weather in Lunapolis?"),
    AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
    HumanMessage(content="How many cheese miners live in Lunapolis?"),
    AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
    HumanMessage(content="Do you think the cheese miners' union will strike?"),
    AIMessage(content="Yes, because they are unhappy with the new president."),
    HumanMessage(
        content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"
    ),
]

config = {"configurable": {"thread_id": ""}}

response = agent.invoke(
    {"messages": messages},
    config=config,
)

print(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is seeking information about Lunapolis, a fictional moon city, including its capital, weather, population of cheese miners, and the potential for a union strike.\n\n## SUMMARY\n- The capital of the moon is identified as Lunapolis.\n- The weather in Lunapolis is described as clear, with a high of 120C and a low of -100C.\n- Lunapolis has a population of 100,000 cheese miners.\n- It is suggested that the cheese miners' union may strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='069f66a2-858b-4ad4-ac8e-4859c49c23f4'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='5bc55c8d-ad79-4bcb-913d-46a8f44bca55'),
              AIMessage(

Notice that it has retained just the last HumanMessage + it's AI response. This is preceeded with a summary of the context.

Now if I print the first (at index 0) of the retained messages, I should see the summarized context. Notice that the model has done a fairly good job of summarizing the context - this provides enough context to the model to progress with the rest of the conversation.

In [9]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user is seeking information about Lunapolis, a fictional moon city, including its capital, weather, population of cheese miners, and the potential for a union strike.

## SUMMARY
- The capital of the moon is identified as Lunapolis.
- The weather in Lunapolis is described as clear, with a high of 120C and a low of -100C.
- Lunapolis has a population of 100,000 cheese miners.
- It is suggested that the cheese miners' union may strike due to dissatisfaction with the new president.

## ARTIFACTS
None

## NEXT STEPS
None


Using this one `SummarizationMiddleware` middleware call, we were able to summarize all the messages in the context, depending on the thresholds we provide.

But what if you didn't want to summarize. What if you wanted to _delete_ all older messages and retain just the most recent ones? Or _maybe_ you want more granular control over which messages are deleted rather than just the oldest X messages. You can do this using a custom middleware function that you _build yourself_!

Custom middleware is implemented using Python functions that are decorated with `@before_XXXX` or `@after_XXXX` decorators to signal that they are called before or after a specific stop in the Agent workflow. For example: `@before_agent` means fire before agent is called. The diagram shows below shows you all the _intervention_ points where you can add custom middleware.

| Core Agent Loop | Where You can _insert_ middleware 'hooks' |
| :-- | :-- |
| The core agent loop wrapped by `create_agent()` call<br/> <div align="center"> <img src="images/01_core_agent_loop.avif" width="250" heigh="100" alt="Core Agent Loop"/> </div> | The middleware _hooks_ that we can apply _before_ and/or _after_ each step <br/> <div align="center"> <img src="images/08_middleware_final.avif" width="250" heigh="100" alt="Middleware"/> </div> |

Following are the decorators we can apply:
* `@before_agent` or `@abefore_agent` - Logic to run _before_ the agent execution starts. The `@abefore_agent` annotates an async function.
* `@before_model` or `@abefore_model` - Logic to run _before_ model is calles. The `@abefore_model` annotates an async function.
* 
@see: https://reference.langchain.com/python/langchain/agents/middleware/types/AgentMiddleware 